# Day 6: CrewAI Agent Crew Test

Tests the 4-agent pipeline on our known artists.

**Architecture:**
- Stage 1 — Crawler Agent: reads cached metadata
- Stage 2 — Fingerprint Analyst: Signals 1, 4, 5 (audio + metadata)
- Stage 3 — Graph Builder: Signals 2, 3, 6 (cadence + ISRC graph)
- Stage 4 — Verdict Agent: weighted synthesis

**API calls:** 0 Spotify | 0 YouTube | 0 Apple Music (all cached)

In [1]:
import sys, os, time
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
import warnings
warnings.filterwarnings('ignore')
from src.agents.crew import run_analysis, run_batch_analysis
print('Agent crew imported OK')

Agent crew imported OK


In [2]:
print('Running pipeline for Relaxing White Noise...')
t0 = time.perf_counter()
rwn_result = run_analysis('6bo3atMVp3qFECNALVwq9N', artist_name='Relaxing White Noise', run_cross_platform=False)
elapsed = time.perf_counter() - t0
print(f'Completed in {elapsed:.1f}s')
print(f'Verdict: {rwn_result["verdict"]} (score={rwn_result["overall_score"]:.3f})')
print(f'Confidence: {rwn_result["confidence"]:.0%}')
for k, v in rwn_result['signal_scores'].items():
    print(f'  {k}: {v:.2f}' if v is not None else f'  {k}: N/A')

2026-04-15 17:48:23.883 | INFO     | src.agents.crew:run_analysis:76 - === Starting analysis pipeline for 6bo3atMVp3qFECNALVwq9N ===
2026-04-15 17:48:23.883 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized


Running pipeline for Relaxing White Noise...


2026-04-15 17:48:25.223 | INFO     | src.agents.crawler_agent:crawl_artist_from_cache:176 - Crawl complete for Relaxing White Noise: 55 albums, 280 tracks
2026-04-15 17:48:25.224 | INFO     | src.agents.crew:run_analysis:95 - Stage 1 (crawl) done in 0.17s
2026-04-15 17:48:25.888 | WARNING  | src.signals.audio_similarity:score_artist:143 - score_artist: no cached track data found for 6bo3atMVp3qFECNALVwq9N in neo4j_full_graph.csv
2026-04-15 17:48:25.888 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized
2026-04-15 17:48:26.964 | INFO     | src.signals.follower_ratio:score_artist:149 - Relaxing White Noise: tracks=280, tracks/day=0.242, avg_dur=185s, suspicion=HIGH (0.76)
2026-04-15 17:48:27.614 | INFO     | src.signals.metadata_similarity:score_cluster:120 - Cluster (3 artists): name_sim=0.000, track_sim=0.041, keyword_score=0.833, genre_jaccard=0.000, suspicion=LOW (0.14)
2026-04-15 17:48:27.615 | INFO     | src.agents.fingerprint_analyst:analyze_fingerprints:188 

Completed in 5.5s
Verdict: SUSPICIOUS (score=0.407)
Confidence: 71%
  s1_audio_similarity: N/A
  s2_cadence_sync: 0.42
  s3_playlist_cooccurrence: 0.00
  s4_follower_ratio: 0.76
  s5_metadata_similarity: 0.14
  s6_graph_density: 0.72
  s7_cross_platform: N/A


In [3]:
print('Running pipeline for Nils Frahm (organic control)...')
t0 = time.perf_counter()
nf_result = run_analysis('5gqhueRUZEa7VDnQt4HODp', artist_name='Nils Frahm', run_cross_platform=False)
elapsed = time.perf_counter() - t0
print(f'Completed in {elapsed:.1f}s')
print(f'Verdict: {nf_result["verdict"]} (score={nf_result["overall_score"]:.3f})')
for k, v in nf_result['signal_scores'].items():
    print(f'  {k}: {v:.2f}' if v is not None else f'  {k}: N/A')

2026-04-15 17:48:29.408 | INFO     | src.agents.crew:run_analysis:76 - === Starting analysis pipeline for 5gqhueRUZEa7VDnQt4HODp ===
2026-04-15 17:48:29.408 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized


Running pipeline for Nils Frahm (organic control)...


2026-04-15 17:48:30.379 | ERROR    | src.agents.crew:_safe_run:307 - Stage 'crawler' failed: 'str' object has no attribute 'get'
2026-04-15 17:48:30.380 | INFO     | src.agents.crew:run_analysis:95 - Stage 1 (crawl) done in 0.00s
2026-04-15 17:48:30.381 | WARNING  | src.signals.audio_similarity:score_artist:143 - score_artist: no cached track data found for 5gqhueRUZEa7VDnQt4HODp in neo4j_full_graph.csv
2026-04-15 17:48:30.381 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized
2026-04-15 17:48:32.126 | INFO     | src.signals.metadata_similarity:score_cluster:120 - Cluster (4 artists): name_sim=0.000, track_sim=0.041, keyword_score=0.625, genre_jaccard=0.000, suspicion=LOW (0.11)
2026-04-15 17:48:32.126 | INFO     | src.agents.fingerprint_analyst:analyze_fingerprints:188 - Fingerprint analysis for 5gqhueRUZEa7VDnQt4HODp: S1=None, S4=0.0, S5=0.108
2026-04-15 17:48:32.127 | INFO     | src.agents.crew:run_analysis:103 - Stage 2 (fingerprint) done in 1.75s
2026-04-15 1

Completed in 4.2s
Verdict: LIKELY_ORGANIC (score=0.026)
  s1_audio_similarity: N/A
  s2_cadence_sync: 0.02
  s3_playlist_cooccurrence: 0.00
  s4_follower_ratio: 0.00
  s5_metadata_similarity: 0.11
  s6_graph_density: 0.00
  s7_cross_platform: N/A


In [4]:
artist_ids = [
    '6bo3atMVp3qFECNALVwq9N',
    '39t4EeLBfpT72UQJVkIeuj',
    '4Wx3ZL6d6p1gVMtwQ2YWsz',
    '5gqhueRUZEa7VDnQt4HODp',
]
print('Running batch analysis on all 4 artists...')
t0 = time.perf_counter()
all_results = run_batch_analysis(artist_ids)
elapsed = time.perf_counter() - t0
print(f'\nBatch completed in {elapsed:.1f}s ({elapsed/len(artist_ids):.1f}s per artist)\n')
print(f"{'Artist':<30} {'Score':>7} {'Verdict':<20} {'Confidence':>12}")
print('-' * 72)
for r in all_results:
    print(f"{r['artist_name']:<30} {r['overall_score']:>7.3f} {r['verdict']:<20} {r['confidence']:>11.0%}")

2026-04-15 17:48:33.578 | INFO     | src.agents.crew:run_batch_analysis:190 - Batch analysis 1/4: 6bo3atMVp3qFECNALVwq9N
2026-04-15 17:48:33.578 | INFO     | src.agents.crew:run_analysis:76 - === Starting analysis pipeline for 6bo3atMVp3qFECNALVwq9N ===
2026-04-15 17:48:33.578 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized


Running batch analysis on all 4 artists...


2026-04-15 17:48:34.730 | INFO     | src.agents.crawler_agent:crawl_artist_from_cache:176 - Crawl complete for Relaxing White Noise: 55 albums, 280 tracks
2026-04-15 17:48:34.731 | INFO     | src.agents.crew:run_analysis:95 - Stage 1 (crawl) done in 0.18s
2026-04-15 17:48:34.731 | WARNING  | src.signals.audio_similarity:score_artist:143 - score_artist: no cached track data found for 6bo3atMVp3qFECNALVwq9N in neo4j_full_graph.csv
2026-04-15 17:48:34.731 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized
2026-04-15 17:48:35.794 | INFO     | src.signals.follower_ratio:score_artist:149 - Relaxing White Noise: tracks=280, tracks/day=0.242, avg_dur=185s, suspicion=HIGH (0.76)
2026-04-15 17:48:36.404 | INFO     | src.signals.metadata_similarity:score_cluster:120 - Cluster (3 artists): name_sim=0.000, track_sim=0.041, keyword_score=0.833, genre_jaccard=0.000, suspicion=LOW (0.14)
2026-04-15 17:48:36.404 | INFO     | src.agents.fingerprint_analyst:analyze_fingerprints:188 


Batch completed in 17.1s (4.3s per artist)

Artist                           Score Verdict                Confidence
------------------------------------------------------------------------
Relaxing White Noise             0.407 SUSPICIOUS                   71%
Meditation Relax Club            0.379 LIKELY_ORGANIC               71%
Calmo                            0.194 LIKELY_ORGANIC               71%
5gqhueRUZEa7VDnQt4HODp           0.026 LIKELY_ORGANIC               71%


In [5]:
from src.signals.verdict import print_report_card
for r in all_results:
    print_report_card(r)


  GHOST DETECTION REPORT: Relaxing White Noise
  Verdict:     SUSPICIOUS
  Score:       0.407  (0=organic, 1=ghost)
  Confidence:  71%  (5/7 signals computed)

  Signal Breakdown:
  S1  Audio Fingerprint  :   N/A  —
  S2  Release Cadence    :   [████████░░░░░░░░░░░░] 0.42  MED
  S3  Playlist Co-occur :   [░░░░░░░░░░░░░░░░░░░░] 0.00  low
  S4  Catalog Ratio      :   [███████████████░░░░░] 0.76  HIGH
  S5  Metadata Sim       :   [██░░░░░░░░░░░░░░░░░░] 0.14  low
  S6  Graph Density (HHI):   [██████████████░░░░░░] 0.72  HIGH
  S7  Cross-Platform     :   N/A  —

  Explanation:
    Relaxing White Noise: SUSPICIOUS (overall_score=0.407)
      MEDIUM cadence suspicion (0.42): irregular release clustering
      HIGH catalog density (0.76): >0.15 tracks/day — suspicious
      HIGH ISRC concentration (0.72): monopolistic production company
      Data unavailable: s1_audio_similarity, s7_cross_platform


  GHOST DETECTION REPORT: Meditation Relax Club
  Verdict:     LIKELY_ORGANIC
  Score:       

In [6]:
# Test CrewAI LLM mode if API key is available
if os.getenv('OPENAI_API_KEY'):
    from src.agents.crew import run_analysis_with_crew
    print('Running CrewAI LLM crew on Relaxing White Noise...')
    t0 = time.perf_counter()
    crew_result = run_analysis_with_crew('6bo3atMVp3qFECNALVwq9N', artist_name='Relaxing White Noise')
    elapsed = time.perf_counter() - t0
    print(f'CrewAI completed in {elapsed:.1f}s')
    print(f'Mode: {crew_result.get("pipeline_mode")}')
    print(f'Verdict: {crew_result["verdict"]} ({crew_result["overall_score"]:.3f})')
    if crew_result.get('crew_narrative'):
        print('\nLLM Narrative (last 400 chars):')
        print(crew_result['crew_narrative'][-400:])
else:
    print('OPENAI_API_KEY not set — skipping LLM crew mode')

2026-04-15 17:48:50.688 | INFO     | src.agents.crew:run_analysis:76 - === Starting analysis pipeline for 6bo3atMVp3qFECNALVwq9N ===
2026-04-15 17:48:50.689 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized


Running CrewAI LLM crew on Relaxing White Noise...


2026-04-15 17:48:51.667 | INFO     | src.agents.crawler_agent:crawl_artist_from_cache:176 - Crawl complete for Relaxing White Noise: 55 albums, 280 tracks
2026-04-15 17:48:51.667 | INFO     | src.agents.crew:run_analysis:95 - Stage 1 (crawl) done in 0.08s
2026-04-15 17:48:51.668 | WARNING  | src.signals.audio_similarity:score_artist:143 - score_artist: no cached track data found for 6bo3atMVp3qFECNALVwq9N in neo4j_full_graph.csv
2026-04-15 17:48:51.668 | INFO     | src.graph.neo4j_client:driver:33 - Neo4j driver initialized
2026-04-15 17:48:52.761 | INFO     | src.signals.follower_ratio:score_artist:149 - Relaxing White Noise: tracks=280, tracks/day=0.242, avg_dur=185s, suspicion=HIGH (0.76)
2026-04-15 17:48:53.367 | INFO     | src.signals.metadata_similarity:score_cluster:120 - Cluster (3 artists): name_sim=0.000, track_sim=0.041, keyword_score=0.833, genre_jaccard=0.000, suspicion=LOW (0.14)
2026-04-15 17:48:53.368 | INFO     | src.agents.fingerprint_analyst:analyze_fingerprints:188 

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cb1bcc82-0b9b-4210-8e56-c39e4ffa08e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Retrieve cached metadata for artist ID: 6bo3atMVp3qFECNALVwq9N. Check artist name, album count, track    │
│  count from cache. Report what data is available.                                                               │
│  ID: 6b0f7a98-cdef-447e-b78c-5c8394fb3caa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: compare_to_baseline                                                                                      │
│  Output: {"artist": "Relaxing White Noise", "comparisons": {"Relaxing White Noise": {"similarity": 0.0,         │
│  "label": "ghost"}, "Meditation Relax Club": {"similarity": 0.0, "label": "ghost"}, "Calmo": {"similarity":     │
│  0.0, "label": "ghost"}, "Nils Frahm": {"similarity": 0.0, "label": "organic"}}, "most_similar_to": "Relaxing   │
│  White Noise", "most_similar_label": "ghost"}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fraud Detection Verdict Analyst                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Final verdict: LIKELY_GHOST                                                                                    │
│                                                                                                                 │
│  Justification:                                                                                                 │
│  Relaxing White Noise exhibits multiple strong indicators commonly associated with ghost artist behavior in     │
│  the music industry. The catalog density is suspiciously high, with a rapid release cadence and a high degree   │
│  of synchronization in bulk drops (42% tracks released on the same day). The ISRC registration graph is highly  │
│  concentrated around a single production company (HHI score 0.7158), further signaling centralized, ghost-like  │
│  control of the catalog. Metadata analysis shows consistency in keyword usage but varied track and artist       │
│  naming, typical of ghost networks aiming to maintain thematic cohesion while avoiding detection. Lack of       │
│  playlist co-occurrence reduces evidence of coordinated promotional activity, but this absence does not negate  │
│  the other strong signals of inauthentic production.                                                            │
│                                                                                                                 │
│  Confidence level: High                                                                                         │
│                                                                                                                 │
│  Actionable recommendation:                                                                                     │
│  Flag this artist for closer human review and potential removal or deeper investigation by platform trust and   │
│  safety teams due to strong evidence of ghost artist behavior, which could be related to fraudulent streaming   │
│  or catalog farming schemes. Monitor for related entities showing similar dense, centralized release patterns.  │
│                                                                                                                 │
│  This conclusion aligns well with internal baseline profiles of known ghost artists in the ambient/noise        │
│  genre.                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize all evidence for Relaxing White Noise (6bo3atMVp3qFECNALVwq9N) into a final verdict. Signal   │
│  scores: {'s1_audio_similarity': None, 's2_cadence_sync': 0.4195, 's3_playlist_cooccurrence': 0.0,              │
│  's4_follower_ratio': 0.7567, 's5_metadata_similarity': 0.1393, 's6_graph_density': 0.7158,                     │
│  's7_cross_platform': None}. Overall score: 0.407. Provide a clear, justified assessment of whether this is a   │
│  ghost artist.                                                                                                  │
│  Agent: Fraud Detection Verdict Analyst                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewAI completed in 27.9s
Mode: crewai_llm
Verdict: SUSPICIOUS (0.407)

LLM Narrative (last 400 chars):
w and potential removal or deeper investigation by platform trust and safety teams due to strong evidence of ghost artist behavior, which could be related to fraudulent streaming or catalog farming schemes. Monitor for related entities showing similar dense, centralized release patterns.

This conclusion aligns well with internal baseline profiles of known ghost artists in the ambient/noise genre.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cb1bcc82-0b9b-4210-8e56-c39e4ffa08e1                                                                       │
│  Final Output: Final verdict: LIKELY_GHOST                                                                      │
│                                                                                                                 │
│  Justification:                                                                                                 │
│  Relaxing White Noise exhibits multiple strong indicators commonly associated with ghost artist behavior in     │
│  the music industry. The catalog density is suspiciously high, with a rapid release cadence and a high degree   │
│  of synchronization in bulk drops (42% tracks released on the same day). The ISRC registration graph is highly  │
│  concentrated around a single production company (HHI score 0.7158), further signaling centralized, ghost-like  │
│  control of the catalog. Metadata analysis shows consistency in keyword usage but varied track and artist       │
│  naming, typical of ghost networks aiming to maintain thematic cohesion while avoiding detection. Lack of       │
│  playlist co-occurrence reduces evidence of coordinated promotional activity, but this absence does not negate  │
│  the other strong signals of inauthentic production.                                                            │
│                                                                                                                 │
│  Confidence level: High                                                                                         │
│                                                                                                                 │
│  Actionable recommendation:                                                                                     │
│  Flag this artist for closer human review and potential removal or deeper investigation by platform trust and   │
│  safety teams due to strong evidence of ghost artist behavior, which could be related to fraudulent streaming   │
│  or catalog farming schemes. Monitor for related entities showing similar dense, centralized release patterns.  │
│                                                                                                                 │
│  This conclusion aligns well with internal baseline profiles of known ghost artists in the ambient/noise        │
│  genre.                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Verify FastAPI backend
import urllib.request, json
BACKEND_URL = 'http://127.0.0.1:8001'

endpoints = [
    ('/', 'Health check'),
    ('/graph/stats', 'Neo4j stats'),
    ('/artists', 'List artists'),
    ('/graph/isrc-clusters', 'ISRC clusters'),
    ('/exercises/summary', 'Exercise findings'),
    ('/graph/neighborhood/6bo3atMVp3qFECNALVwq9N', 'RWN neighborhood'),
]

print(f'Testing FastAPI backend at {BACKEND_URL}\n')
for path, label in endpoints:
    try:
        with urllib.request.urlopen(f'{BACKEND_URL}{path}', timeout=10) as r:
            data = json.loads(r.read())
            print(f'OK  GET {path} — {label}')
    except Exception as e:
        print(f'ERR GET {path} — {e}')

Testing FastAPI backend at http://127.0.0.1:8001

OK  GET / — Health check
OK  GET /graph/stats — Neo4j stats
OK  GET /artists — List artists
OK  GET /graph/isrc-clusters — ISRC clusters
OK  GET /exercises/summary — Exercise findings
OK  GET /graph/neighborhood/6bo3atMVp3qFECNALVwq9N — RWN neighborhood


In [8]:
print('=== TIMING SUMMARY ===')
for r in all_results:
    t = r.get('timing', {})
    print(f"{r['artist_name']:<30} total={t.get('total_seconds',0):.1f}s")
print('\nAll artists under 30s. No Spotify API calls. All from Neo4j + cache.')

=== TIMING SUMMARY ===
Relaxing White Noise           total=4.5s
Meditation Relax Club          total=4.2s
Calmo                          total=4.2s
5gqhueRUZEa7VDnQt4HODp         total=4.2s

All artists under 30s. No Spotify API calls. All from Neo4j + cache.
